# 🧠 Practical Application of Transpose Convolution

> **Module:** 03 — Deep Learning with Keras and TensorFlow  
> **Topic:** Transpose convolution (`Conv2DTranspose`), image reconstruction, autoencoder-style models  
> **Architecture:** Encoder (Conv2D) → Decoder (Conv2DTranspose)

---

## 📋 Overview

I build a minimal **encoder–decoder** for image reconstruction using `Conv2DTranspose`. The model compresses a 28×28 image through a Conv2D layer, then expands it back using transpose convolution, learning to reproduce the original pixel values.

| Part | Step | Description |
|---|---|---|
| Part 1 | Define architecture | Input → Conv2D(32) → Conv2DTranspose(1) |
| Part 2 | Compile & train | MSE loss, Adam, synthetic random images |
| Part 3 | Evaluate & visualise | Test loss + original vs. reconstructed grid |
| Exercises | Kernel size, Dropout, activations | Solved with comparisons |

## 🧩 Theory

### What is transpose convolution?

A standard `Conv2D` maps a large spatial input to a smaller feature map (downsampling). `Conv2DTranspose` does the reverse: maps a small feature map to a larger output (upsampling) using **learned weights**.

**Telecom / RF analogy 📡:** Like a matched filter followed by upsampling in a receiver chain. Regular convolution correlates with a kernel (like a correlator bank). Transpose convolution expands back to the signal domain using learned interpolation — like a pulse-shaping filter that reconstructs the continuous waveform from discrete samples, but with learned (not fixed sinc) weights.

### Output size formula

For `Conv2DTranspose` with stride $s$, `padding='same'`:
$$H_{out} = H_{in} \times s$$

With `stride=1` (default): $H_{out} = H_{in}$ — same spatial size maintained.

### Transpose conv vs UpSampling2D

| Method | Weights | Learns upsampling? | Typical use |
|---|---|---|---|
| `UpSampling2D` | None | ❌ Fixed interpolation | Fast, simple decoders |
| `Conv2DTranspose` | Trainable | ✅ Learned | GANs, autoencoders, segmentation |

### Reconstruction loss (MSE)

$$\mathcal{L}_{\text{MSE}} = \frac{1}{N \cdot H \cdot W \cdot C} \sum_{i,h,w,c} (x_{i,h,w,c} - \hat{x}_{i,h,w,c})^2$$

### Encoder–decoder architecture

```
Input (28×28×1)
  ↓  Conv2D(32, 3×3, ReLU, same)      → 28×28×32   [encode]
  ↓  Conv2DTranspose(1, 3×3, σ, same)  → 28×28×1   [decode]
Output (28×28×1)
```
Target = Input: the network reproduces its own input, learning an efficient intermediate representation.

## ⚙️ Part 0 — Setup

In [ ]:
import warnings; warnings.simplefilter('ignore')
!pip install tensorflow==2.16.2 matplotlib --quiet

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, UpSampling2D, Dropout
print(f"TensorFlow: {tf.__version__}")

## 🏗️ Part 1 — Define the Architecture

Input: 28×28×1 grayscale images (MNIST shape). Tensor shape: $(N, 28, 28, 1)$.

In [ ]:
input_layer = Input(shape=(28, 28, 1))
print(f"Input layer shape: {input_layer.shape}")

**Encoder:** `Conv2D(32, 3×3, ReLU, same)` extracts 32 feature maps — output shape $(N, 28, 28, 32)$.

**Decoder:** `Conv2DTranspose(1, 3×3, sigmoid, same)` maps 32 channels back to a single-channel 28×28 output. Sigmoid constrains output to $(0, 1)$, matching normalised pixel range.

In [ ]:
conv_layer = Conv2D(
    filters=32, kernel_size=(3, 3), activation='relu', padding='same'
)(input_layer)

transpose_conv_layer = Conv2DTranspose(
    filters=1, kernel_size=(3, 3), activation='sigmoid', padding='same'
)(conv_layer)

print(f"Encoder output: {conv_layer.shape}")
print(f"Decoder output: {transpose_conv_layer.shape}")

In [ ]:
model = Model(inputs=input_layer, outputs=transpose_conv_layer)
model.summary()

## 📐 Part 2 — Compile & Train

Loss: MSE — $\mathcal{L} = \frac{1}{N}\sum \|x - \hat{x}\|^2$. No accuracy metric — this is pixel-level regression, not classification.

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')
print('Model compiled ✅  optimizer=Adam  loss=MSE')

Synthetic data: 1000 random images $X \sim \mathcal{U}(0,1)^{28 \times 28}$. Target = input (reconstruction task).

In [ ]:
X_train = np.random.rand(1000, 28, 28, 1)
y_train = X_train  # reconstruction: target = input
print(f"Train shape: {X_train.shape}")

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train MSE',      color='steelblue')
plt.plot(history.history['val_loss'], label='Validation MSE', color='tomato', linestyle='--')
plt.title('📉 MSE Loss — Training vs Validation')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 📊 Part 3 — Evaluate & Visualise

In [ ]:
X_test = np.random.rand(200, 28, 28, 1)
y_test = X_test
test_loss = model.evaluate(X_test, y_test, verbose=0)
print(f'🎯 Test MSE: {test_loss:.6f}  |  RMSE: {test_loss**0.5:.6f}')

In [ ]:
y_pred = model.predict(X_test, verbose=0)
n = 10
fig, axes = plt.subplots(2, n, figsize=(20, 4))
for i in range(n):
    axes[0][i].imshow(X_test[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[0][i].set_title('Original', fontsize=8); axes[0][i].axis('off')
    axes[1][i].imshow(y_pred[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1][i].set_title('Reconstructed', fontsize=8); axes[1][i].axis('off')
plt.suptitle('📊 Original vs. Reconstructed Images', fontsize=12)
plt.tight_layout(); plt.show()

# Pixel error map
error_map = np.abs(X_test[0].reshape(28,28) - y_pred[0].reshape(28,28))
plt.figure(figsize=(4,4))
plt.imshow(error_map, cmap='hot'); plt.colorbar(label='|Original - Reconstructed|')
plt.title('🔥 Pixel Error Map (sample 0)'); plt.axis('off')
plt.tight_layout(); plt.show()

---

## 🔢 Exercises — Solved

### Exercise 1 — Larger Kernel Size (5×5)

Larger kernel = wider receptive field. Params: $32 \times (5\times5\times1) + 32 = 832$ vs $320$ for 3×3.

In [ ]:
input_ex1 = Input(shape=(28, 28, 1))
conv_ex1  = Conv2D(32, (5,5), activation='relu',    padding='same')(input_ex1)
deconv_ex1 = Conv2DTranspose(1, (5,5), activation='sigmoid', padding='same')(conv_ex1)
model_ex1 = Model(inputs=input_ex1, outputs=deconv_ex1)
model_ex1.compile(optimizer='adam', loss='mean_squared_error')
history_ex1 = model_ex1.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
loss_ex1 = model_ex1.evaluate(X_test, y_test, verbose=0)
print(f'✅ Test MSE (5×5): {loss_ex1:.6f}  vs baseline (3×3): {test_loss:.6f}')

plt.figure(figsize=(8,4))
plt.plot(history.history['loss'],     label='3×3 train', color='steelblue')
plt.plot(history_ex1.history['loss'], label='5×5 train', color='darkorange')
plt.plot(history.history['val_loss'],     linestyle='--', color='steelblue',  label='3×3 val')
plt.plot(history_ex1.history['val_loss'], linestyle='--', color='darkorange', label='5×5 val')
plt.title('✅ Ex 1 — Kernel 3×3 vs 5×5'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Exercise 2 — Add Dropout(0.5)

Dropout zeros 50% of encoder features per step: $h'_j = h_j \cdot \text{Bernoulli}(0.5)$.  
Forces decoder to reconstruct from incomplete information — more robust features.

In [ ]:
input_ex2  = Input(shape=(28, 28, 1))
conv_ex2   = Conv2D(32, (3,3), activation='relu',    padding='same')(input_ex2)
drop_ex2   = Dropout(0.5)(conv_ex2)
deconv_ex2 = Conv2DTranspose(1, (3,3), activation='sigmoid', padding='same')(drop_ex2)
model_ex2 = Model(inputs=input_ex2, outputs=deconv_ex2)
model_ex2.compile(optimizer='adam', loss='mean_squared_error')
history_ex2 = model_ex2.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
loss_ex2 = model_ex2.evaluate(X_test, y_test, verbose=0)
print(f'✅ Test MSE (Dropout): {loss_ex2:.6f}  vs baseline: {test_loss:.6f}')

plt.figure(figsize=(8,4))
plt.plot(history.history['val_loss'],     label='No Dropout val',   color='steelblue', linestyle='--')
plt.plot(history_ex2.history['val_loss'], label='Dropout(0.5) val', color='purple',    linestyle='--')
plt.title('✅ Ex 2 — Dropout(0.5) Effect'); plt.xlabel('Epoch'); plt.ylabel('Val MSE')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Exercise 3 — Tanh Activation

| Activation | Range | Note |
|---|---|---|
| ReLU | $[0, \infty)$ | May have dead neurons |
| Sigmoid | $(0, 1)$ | Saturates at extremes |
| Tanh | $(-1, 1)$ | Zero-centred; requires re-scaled targets |

In [ ]:
# Re-scale to [-1, 1] to match tanh output
X_train_tanh = 2.0 * X_train - 1.0
X_test_tanh  = 2.0 * X_test  - 1.0

input_ex3  = Input(shape=(28, 28, 1))
conv_ex3   = Conv2D(32, (3,3), activation='tanh', padding='same')(input_ex3)
deconv_ex3 = Conv2DTranspose(1, (3,3), activation='tanh', padding='same')(conv_ex3)
model_ex3 = Model(inputs=input_ex3, outputs=deconv_ex3)
model_ex3.compile(optimizer='adam', loss='mean_squared_error')
history_ex3 = model_ex3.fit(X_train_tanh, X_train_tanh, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
loss_ex3 = model_ex3.evaluate(X_test_tanh, X_test_tanh, verbose=0)
print(f'✅ Test MSE (tanh): {loss_ex3:.6f}')

y_pred_tanh = model_ex3.predict(X_test_tanh[:5], verbose=0)
y_disp = (y_pred_tanh + 1.0) / 2.0  # map back to [0,1] for display
fig, axes = plt.subplots(2, 5, figsize=(12, 4))
for i in range(5):
    axes[0][i].imshow(X_test[i].reshape(28,28), cmap='gray', vmin=0, vmax=1); axes[0][i].axis('off'); axes[0][i].set_title('Original', fontsize=8)
    axes[1][i].imshow(y_disp[i].reshape(28,28),  cmap='gray', vmin=0, vmax=1); axes[1][i].axis('off'); axes[1][i].set_title('tanh recon.', fontsize=8)
plt.suptitle('✅ Ex 3 — Tanh Activation Reconstruction', fontsize=12)
plt.tight_layout(); plt.show()

---

## 📊 Summary

| Component | Config | Purpose |
|---|---|---|
| `Conv2D(32, 3×3, ReLU, same)` | 320 params | Extract 32 feature maps |
| `Conv2DTranspose(1, 3×3, sigmoid, same)` | 289 params | Reconstruct single-channel image |
| Loss: MSE | $\frac{1}{N}\|x-\hat{x}\|^2$ | Pixel-level fidelity |
| Optimizer: Adam | LR=0.001 | Fast convergence |

**Key rules:**
- For reconstruction: **target = input** (unsupervised)
- `padding='same'` preserves spatial dimensions through both layers
- Decoder activation must match data range: sigmoid for $[0,1]$, tanh for $[-1,1]$
- `Conv2DTranspose` with `stride=1` is a feature mixer; `stride>1` gives actual spatial upsampling

---

## 🧪 Sandbox

In [ ]:
# SANDBOX 1: Strided autoencoder — proper bottleneck (28×28 → 7×7 → 28×28)
# Telecom analogy: source coding (compress) → channel → source decoding (expand)
inp = Input(shape=(28, 28, 1))
e1  = Conv2D(32, (3,3), activation='relu',    padding='same', strides=2)(inp)   # 28→14
e2  = Conv2D(64, (3,3), activation='relu',    padding='same', strides=2)(e1)    # 14→7
d1  = Conv2DTranspose(32, (3,3), activation='relu',    padding='same', strides=2)(e2)  # 7→14
d2  = Conv2DTranspose(1,  (3,3), activation='sigmoid', padding='same', strides=2)(d1)  # 14→28
ae  = Model(inputs=inp, outputs=d2)
ae.compile(optimizer='adam', loss='mse')
ae.summary()
hist_ae = ae.fit(X_train, X_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
print(f'Strided AE test MSE: {ae.evaluate(X_test, X_test, verbose=0):.6f}')

In [ ]:
# SANDBOX 2: UpSampling2D + Conv2D vs Conv2DTranspose
inp_up = Input(shape=(28, 28, 1))
c_up   = Conv2D(32, (3,3), activation='relu', padding='same')(inp_up)
up     = UpSampling2D(size=(1,1))(c_up)
out_up = Conv2D(1, (3,3), activation='sigmoid', padding='same')(up)
model_up = Model(inputs=inp_up, outputs=out_up)
model_up.compile(optimizer='adam', loss='mse')
model_up.fit(X_train, X_train, epochs=10, batch_size=32, validation_split=0.2, verbose=0)
loss_up = model_up.evaluate(X_test, X_test, verbose=0)
print(f'UpSampling+Conv MSE: {loss_up:.6f}')
print(f'Conv2DTranspose MSE: {test_loss:.6f}')

In [ ]:
# SANDBOX 3: Anomaly detection via reconstruction error
# Telecom analogy: model trained on noise flags structured patterns (interferers) by high MSE
X_anomaly = np.zeros((20, 28, 28, 1))
X_anomaly[:, 10:18, 10:18, :] = 1.0  # bright square

y_normal  = model.predict(X_test[:20],  verbose=0)
y_anomaly = model.predict(X_anomaly,    verbose=0)
mse_normal  = np.mean((X_test[:20]  - y_normal)**2,  axis=(1,2,3))
mse_anomaly = np.mean((X_anomaly    - y_anomaly)**2, axis=(1,2,3))

plt.figure(figsize=(8,4))
plt.hist(mse_normal,  bins=10, alpha=0.6, label='Normal (random)',    color='steelblue')
plt.hist(mse_anomaly, bins=10, alpha=0.6, label='Anomalous (square)', color='tomato')
plt.title('🧪 Sandbox 3 — Anomaly Detection via Reconstruction Error')
plt.xlabel('Per-sample MSE'); plt.ylabel('Count'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Mean MSE normal:    {mse_normal.mean():.6f}')
print(f'Mean MSE anomalous: {mse_anomaly.mean():.6f}')